In [1]:
# =============================================================================
# RECOMPUTE EVERY REPORTED NUMBER  ->  readable tables + figures + files
# =============================================================================
# KAGGLE: Accelerator None (CPU), Internet Off, Add Input = your results Dataset.
# Every section prints a real table. In a notebook they render as HTML tables;
# in a plain console they print aligned. CSV and JSON copies are written too.
# =============================================================================

import os, re, json, glob, math
from collections import Counter
from math import comb

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SEARCH = ["/kaggle/input", "/kaggle/working", "."]
R, TABLES = {}, {}

try:                                    # pretty tables in a notebook
    from IPython.display import display
    IN_NB = True
except Exception:
    IN_NB = False


def show(title, df, note=None):
    """Print a titled table. Renders as HTML in a notebook, aligned text elsewhere."""
    TABLES[title] = df
    print("\n" + "=" * 86)
    print(title.upper())
    print("=" * 86)
    if IN_NB:
        display(df)
    else:
        print(df.to_string(index=False))
    if note:
        print(f"  note: {note}")


def find(*patterns):
    for root in SEARCH:
        for pat in patterns:
            hits = sorted(glob.glob(f"{root}/**/{pat}", recursive=True))
            if hits:
                return hits[0]
    return None


def tb(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1"])


def mcnemar(b, c):
    n = b + c
    if n == 0:
        return 1.0
    return min(2 * sum(comb(n, i) for i in range(max(b, c), n + 1)) * 0.5 ** n, 1.0)


def wilson(k, n):
    p = k / n
    se = math.sqrt(p * (1 - p) / n)
    return p - 1.96 * se, p + 1.96 * se


def pfmt(p):
    return "<1e-9" if p < 1e-9 else (f"{p:.1e}" if p < 0.001 else f"{p:.3f}")


def arm(path, label, spec):
    d = pd.read_csv(path)
    if "error" in d.columns:
        d = d[d["error"].isna() | (d["error"].astype(str).str.strip() == "")]
    g, b = tb(d.guided_correct), tb(d.baseline_correct)
    n = len(d)
    go, bo = int((g & ~b).sum()), int((b & ~g).sum())
    return {"Executor": label, "Specialised": "yes" if spec else "no", "n": n,
            "No plan": f"{b.mean()*100:.2f}% ({int(b.sum())})",
            "Planned": f"{g.mean()*100:.2f}% ({int(g.sum())})",
            "Net pp": f"{(g.mean()-b.mean())*100:+.2f}",
            "Net probs": f"{int(g.sum()-b.sum()):+d}",
            "Wins:losses": f"{go}:{bo}", "p": pfmt(mcnemar(go, bo)),
            "Significant": "yes" if mcnemar(go, bo) < 0.05 else "NO",
            "Source": os.path.basename(path)}, d


# ---------------------------------------------- 1. main effect on GSM8K ---
rows, FRAMES = [], {}
for pats, label, spec in [
        (("grad_qwen2*0?5B*.csv", "grad*0.5B*.csv"), "Qwen2.5-0.5B", False),
        (("grad_qwen2*1?5B*.csv", "grad*1.5B*.csv"), "Qwen2.5-1.5B", False),
        (("grad_qwen2*3B*.csv",), "Qwen2.5-3B", False),
        (("twoslm_full_1319*.csv", "twoslm*.csv"), "Qwen2.5-Math-1.5B", True)]:
    f = find(*pats)
    if f:
        r, FRAMES[label] = arm(f, label, spec)
        rows.append(r)

if not any(r["Executor"] == "Qwen2.5-1.5B" for r in rows):
    b, go, bo, n = 600, 310, 172, 1319        # CSV lost; rebuild from discordants
    g = b + go - bo
    rows.insert(1, {"Executor": "Qwen2.5-1.5B", "Specialised": "no", "n": n,
                    "No plan": f"{b/n*100:.2f}% ({b})", "Planned": f"{g/n*100:.2f}% ({g})",
                    "Net pp": f"{(g-b)/n*100:+.2f}", "Net probs": f"{g-b:+d}",
                    "Wins:losses": f"{go}:{bo}", "p": pfmt(mcnemar(go, bo)),
                    "Significant": "yes", "Source": "RECONSTRUCTED 600+310-172"})

df_main = pd.DataFrame(rows)
show("Table 1  guidance effect by executor, GSM8K test",
     df_main,
     "the 3B model is stronger than the 1.5B and gains MORE, so this is not "
     "compensation for weakness")

# -------------------------------------------------------------- 2. OOD ---
rows = []
for pat, label in [("ood_svamp_general.csv", "SVAMP"),
                   ("ood_multiarith_general.csv", "MultiArith")]:
    f = find(pat)
    if not f:
        continue
    r, _ = arm(f, label, False)
    r["Executor"] = label
    d = pd.read_csv(f)
    p0, n = tb(d.baseline_correct).mean(), len(d)
    mde = 1.96 * math.sqrt(2 * p0 * (1 - p0) / n) * 100
    net = abs(float(r["Net pp"]))
    r["MDE pp"] = f"{mde:.1f}"
    r["Verdict"] = "underpowered" if net < mde else "adequately powered"
    rows.append(r)
if rows:
    df_ood = (pd.DataFrame(rows)
                .rename(columns={"Executor": "Dataset"})
                .drop(columns=["Specialised"]))
    show("Table 2  out-of-distribution, executor held at Qwen2.5-1.5B", df_ood,
         "a non-significant result below the MDE is inconclusive, not a null")

# ----------------------------------------------------- 3. planner quality ---
rows = []
f = find("twoslm_full_1319*.csv", "twoslm*.csv")
if f:
    d = pd.read_csv(f)
    rows.append({"Dataset": "GSM8K (in-dist.)", "n": len(d),
                 "Format ok": f"{tb(d.plan_format_ok).mean()*100:.2f}%",
                 "Ops in vocab": f"{tb(d.plan_ops_valid).mean()*100:.2f}%",
                 "Arith. leak": f"{tb(d.plan_has_math).mean()*100:.2f}%",
                 "Mean steps": f"{d.plan_steps.mean():.2f}",
                 "Source": os.path.basename(f)})
    ops = [o for s in d.plan_operators.dropna() for o in str(s).split()]
    cnt, tot = Counter(ops), len(ops)
    df_ops = pd.DataFrame([{"Operator": k, "Steps": v, "Share": f"{v/tot*100:.2f}%"}
                           for k, v in cnt.most_common()])
# the OOD plan files use format_ok / ops_valid / has_math; GSM8K prefixes plan_*
for pat, label in [("plans_svamp.csv", "SVAMP (unseen)"),
                   ("plans_multiarith.csv", "MultiArith (unseen)")]:
    f = find(pat)
    if f:
        p = pd.read_csv(f)
        rows.append({"Dataset": label, "n": len(p),
                     "Format ok": f"{tb(p.format_ok).mean()*100:.2f}%",
                     "Ops in vocab": f"{tb(p.ops_valid).mean()*100:.2f}%",
                     "Arith. leak": f"{tb(p.has_math).mean()*100:.2f}%",
                     "Mean steps": "", "Source": os.path.basename(f)})
if rows:
    df_pq = pd.DataFrame(rows)
    show("Table 3  planner quality", df_pq,
         "zero arithmetic leakage everywhere is what the architecture depends on")
    if "df_ops" in dir():
        show(f"Table 3b  operator distribution ({tot} tagged steps)", df_ops)

# ------------------------------------------- 4. accuracy by plan length ---
BINS = [(1, 2, "1-2"), (3, 3, "3"), (4, 4, "4"), (5, 99, "5+")]
rows = []
for label, d in FRAMES.items():
    if "plan" not in d.columns:
        continue
    ns = d.plan.astype(str).str.count(r"Step\s*\d")
    for lo, hi, name in BINS:
        s = d[(ns >= lo) & (ns <= hi)]
        if len(s) < 20:
            continue
        b, g = tb(s.baseline_correct).mean() * 100, tb(s.guided_correct).mean() * 100
        rows.append({"Executor": label, "Steps": name, "n": len(s),
                     "No plan": f"{b:.1f}%", "Planned": f"{g:.1f}%",
                     "Gain pp": f"{g-b:+.1f}"})
if rows:
    df_cx = pd.DataFrame(rows)
    show("Table 4  accuracy by plan length", df_cx,
         "the specialised model gains little at every length and goes negative at 5+")
    show("Table 4b  gain in pp, pivoted",
         df_cx.assign(g=lambda x: x["Gain pp"].astype(float))
              .pivot(index="Steps", columns="Executor", values="g")
              .reindex(["1-2", "3", "4", "5+"]).reset_index())

# ------------------------------------------------------ 5. single model ---
rows = []
for pat, label in [("slm1_compressed_full1319.csv", "equations only / Qwen-Math"),
                   ("phase3_gsm8k.csv", "planner-taught / 1.5B"),
                   ("ood_svamp_1slm.csv", "equations only / Qwen-Math, SVAMP"),
                   ("ood_multiarith_1slm.csv", "equations only / Qwen-Math, MultiArith")]:
    f = find(pat)
    if not f:
        continue
    d = pd.read_csv(f)
    c = tb(d.correct)
    lo, hi = wilson(int(c.sum()), len(d))
    rows.append({"Configuration": label, "n": len(d), "Correct": int(c.sum()),
                 "Accuracy": f"{c.mean()*100:.2f}%",
                 "95% CI": f"[{lo*100:.1f}, {hi*100:.1f}]",
                 "Top parse rule": (d.strategy.value_counts().index[0]
                                    if "strategy" in d.columns else ""),
                 "Source": os.path.basename(f)})
f = find("phase3_summary.json")
if f:
    j = json.load(open(f)); g = j["gsm8k"]
    rows.append({"Configuration": "planner-taught / 1.5B (JSON)", "n": g["n"],
                 "Correct": g["correct"], "Accuracy": f"{g['acc']*100:.2f}%",
                 "95% CI": f"[{g['ci'][0]*100:.1f}, {g['ci'][1]*100:.1f}]",
                 "Top parse rule": max(g["strategies"], key=g["strategies"].get),
                 "Source": "phase3_summary.json"})
    R["phase3"] = j
if rows:
    show("Table 5  single-model results", pd.DataFrame(rows))

# ---------------------------------------------- 6. cost of a correct answer ---
COST = [("one model, equations", 698, 41, 4.13), ("one model, planner-taught", 738, 116, 11.77),
        ("one model, teacher-taught", 718, 118, 12.14), ("executor alone, no plan", 600, 130, 15.57),
        ("planner + executor", 738, 230, 23.20)]
df_cost = pd.DataFrame([{"Configuration": n, "Accuracy": f"{s/1319*100:.2f}% ({s})",
                         "Tokens": t, "s/item": f"{l:.2f}",
                         "s/correct": f"{l/(s/1319):.1f}"} for n, s, t, l in COST])
show("Table 6  cost of one correct answer, Qwen2.5-1.5B base", df_cost,
     "planner-taught and planner+executor tie at 738; everything right of "
     "Accuracy favours the single model")

# ------------------------------------------------- 7. latency + agreement ---
f = find("latency_matched.json")
if f:
    j = json.load(open(f)); R["latency"] = j
    show("Table 7  measured latency, fp16 sequential batch 1",
         pd.DataFrame([{"Configuration": k, "Mean s": f"{v['mean_s']:.2f}",
                        "Median s": f"{v['median_s']:.2f}"} for k, v in j.items()]))
f = find("planner_teacher_agreement.json")
if f:
    j = json.load(open(f)); R["agreement"] = j
    show("Table 8  planner vs teacher decomposition",
         pd.DataFrame([{"Measure": k.replace("_", " "),
                        "Value": f"{v*100:.2f}%" if isinstance(v, float) and v <= 1 else v}
                       for k, v in j.items()]),
         "45% exact match, yet the student trained on these plans is no worse")

# ------------------------------------------------------------ 8. figures ---
plt.rcParams.update({"font.size": 9, "font.family": "serif",
                     "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 200})


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUT}/{name}.{ext}", bbox_inches="tight")
    plt.close(fig)


if rows and "df_cx" in dir():
    order = [k for k in ["Qwen2.5-3B", "Qwen2.5-1.5B", "Qwen2.5-0.5B",
                         "Qwen2.5-Math-1.5B"] if k in df_cx.Executor.unique()]
    labels = ["1-2", "3", "4", "5+"]
    x = np.arange(len(labels)); w = 0.8 / max(len(order), 1)
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    for i, k in enumerate(order):
        sub = df_cx[df_cx.Executor == k].set_index("Steps")
        vals = [float(sub.loc[b, "Gain pp"]) if b in sub.index else 0 for b in labels]
        bars = ax.bar(x + (i - (len(order) - 1) / 2) * w, vals, w, label=k,
                      color=["#1f4e79", "#4472c4", "#9dc3e6", "#c00000"][i % 4])
        ax.bar_label(bars, fmt="%.1f", fontsize=6, padding=1)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_xlabel("Steps in the generated plan"); ax.set_ylabel("Gain from a plan (pp)")
    ax.axhline(0, color="k", lw=0.7); ax.legend(frameon=False, fontsize=7)
    fig.tight_layout(); save(fig, "fig_complexity")

fig, ax = plt.subplots(figsize=(4.6, 3.1))
for _, r in df_main.iterrows():
    xx = float(r["No plan"].split("%")[0]); yy = float(r["Net pp"])
    sp = r["Specialised"] == "yes"
    ax.scatter(xx, yy, s=70, marker="s" if sp else "o",
               color="#c00000" if sp else "#1f4e79", zorder=3)
    ax.annotate(r["Executor"].replace("Qwen2.5-", ""), (xx, yy),
                textcoords="offset points", xytext=(5, 4), fontsize=7)
ax.axhspan(-2, 3.3, color="grey", alpha=0.18)
ax.set_xlabel("Unaided accuracy on GSM8K (%)"); ax.set_ylabel("Gain from a plan (pp)")
fig.tight_layout(); save(fig, "fig_gradient")

fig, ax = plt.subplots(figsize=(4.6, 3.1))
for nm, s, t, l in COST:
    front = nm in ("one model, equations", "one model, planner-taught")
    ax.scatter(l, s / 1319 * 100, s=110 if front else 55, marker="*" if front else "o",
               color="#c00000" if front else "#888888", zorder=3)
    ax.annotate(nm, (l, s / 1319 * 100), textcoords="offset points",
                xytext=(5, -9), fontsize=6.5)
ax.set_xlabel("Time per request (s)"); ax.set_ylabel("Solved on GSM8K (%)"); ax.set_xlim(0, 27)
fig.tight_layout(); save(fig, "fig_frontier")

# -------------------------------------------------------------- 9. write ---
for title, df in TABLES.items():
    slug = re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:50]
    df.to_csv(f"{OUT}/{slug}.csv", index=False)
R["tables"] = {t: d.to_dict("records") for t, d in TABLES.items()}
json.dump(R, open(f"{OUT}/results_all.json", "w"), indent=2, default=str)

show("Files written", pd.DataFrame(
    [{"File": f, "KB": round(os.path.getsize(f"{OUT}/{f}") / 1024, 1)}
     for f in sorted(os.listdir(OUT))
     if f.endswith((".csv", ".json", ".png", ".pdf"))]))


TABLE 1  GUIDANCE EFFECT BY EXECUTOR, GSM8K TEST


,Executor,Specialised,n,No plan,Planned,Net pp,Net probs,Wins:losses,p,Significant,Source
0,Qwen2.5-0.5B,no,1319,22.59% (298),31.92% (421),+9.33,+123,263:140,<1e-9,yes,grad_qwen2.5-0.5B.csv
1,Qwen2.5-1.5B,no,1319,45.49% (600),55.95% (738),+10.46,+138,310:172,<1e-9,yes,RECONSTRUCTED 600+310-172
2,Qwen2.5-3B,no,1319,53.53% (706),68.99% (910),+15.47,+204,327:123,<1e-9,yes,grad_qwen2.5-3B.csv
3,Qwen2.5-Math-1.5B,yes,1319,69.45% (916),71.65% (945),+2.20,+29,196:167,0.142,NO,twoslm_full_1319 (1).csv


  note: the 3B model is stronger than the 1.5B and gains MORE, so this is not compensation for weakness

TABLE 2  OUT-OF-DISTRIBUTION, EXECUTOR HELD AT QWEN2.5-1.5B


,Dataset,n,No plan,Planned,Net pp,Net probs,Wins:losses,p,Significant,Source,MDE pp,Verdict
0,SVAMP,300,63.33% (190),65.00% (195),+1.67,+5,50:45,0.682,NO,ood_svamp_general.csv,7.7,underpowered
1,MultiArith,180,82.22% (148),91.11% (164),+8.89,+16,27:11,0.014,yes,ood_multiarith_general.csv,7.9,adequately powered


  note: a non-significant result below the MDE is inconclusive, not a null

TABLE 3  PLANNER QUALITY


...

,Configuration,Accuracy,Tokens,s/item,s/correct
0,"one model, equations",52.92% (698),41,4.13,7.8
1,"one model, planner-taught",55.95% (738),116,11.77,21.0
2,"one model, teacher-taught",54.44% (718),118,12.14,22.3
3,"executor alone, no plan",45.49% (600),130,15.57,34.2
4,planner + executor,55.95% (738),230,23.20,41.5


  note: planner-taught and planner+executor tie at 738; everything right of Accuracy favours the single model

TABLE 7  MEASURED LATENCY, FP16 SEQUENTIAL BATCH 1


,Configuration,Mean s,Median s
0,Phase A (planner) alone,9.79,8.89
1,2-SLM guided (Qwen-Math),24.75,23.14
2,Phase B alone (Qwen-Math),12.37,10.94
3,2-SLM guided (Qwen general),23.20,20.56
4,Phase B alone (Qwen general),15.57,12.90
5,1-SLM compressed,3.12,2.99



FILES WRITTEN


,File,KB
0,fig_frontier.pdf,16.2
1,fig_frontier.png,46.3
2,fig_gradient.pdf,16.2
3,fig_gradient.png,37.8
4,results_all.json,9.2
5,table_1_guidance_effect_by_executor_gsm8k_test...,0.5
6,table_2_out_of_distribution_executor_held_at_q...,0.3
7,table_3_planner_quality.csv,0.3
8,table_3b_operator_distribution_4492_tagged_ste...,0.2
9,table_4_accuracy_by_plan_length.csv,0.5
